In [ ]:
import numpy as np
import pandas as pd
from icecream import ic
from tqdm import tqdm
import time
from IPython.display import clear_output

import requests
from nba_api.stats.library.http import NBAStatsHTTP

from nba_api.stats.endpoints import (
    boxscoreadvancedv3,
    boxscoretraditionalv2,
    boxscoretraditionalv3,
    boxscorefourfactorsv3,
    boxscorehustlev2,
    boxscoremiscv3,
    boxscoreplayertrackv3,
    boxscorescoringv3,
    leaguegamelog,
    leaguedashplayerstats,
    leaguedashteamstats,
    boxscoredefensivev2,
    boxscorematchupsv3,
    boxscoreusagev3,
)

pd.options.mode.chained_assignment =  None

box_DIR = "../../data/box/"
shiny_DIR = "../../data/shiny/"

In [ ]:
def get_gameids_PS(season, name):
    df = pd.read_parquet(
        box_DIR + "NBA_Box_T_" + "Base" + "_" + season + "_PS.parquet"
    )
    game_ids1 = df["GAME_ID"].tolist()
    game_ids1 = np.unique(game_ids1)
    try:
        dfr2 = pd.read_parquet(box_DIR + "NBA_Box_P_" + name + "_" + season + "_PS.parquet")
        if name != "Matchup":
            dfr1 = pd.read_parquet(box_DIR + "NBA_Box_T_" + name + "_" + season + "_PS.parquet")
            game_ids3 = dfr1["gameId"].tolist()
        else:
            dfr1 = pd.DataFrame()
            game_ids3 = dfr2["gameId"].tolist()
        game_ids2 = np.unique(game_ids3)
        game_ids2 = ["00" + str(s) for s in game_ids2]
        game_ids = list(set(game_ids1).difference(game_ids2))
    except Exception as error:
        print(error)
        game_ids = game_ids1
        dfr1 = pd.DataFrame()
        dfr2 = pd.DataFrame()
    return game_ids, dfr1, dfr2

def get_game_box(game_id,fun):
    t1 = time.perf_counter()
    stats = fun(game_id=game_id)
    df = stats.get_data_frames()
    t2 = time.perf_counter() - t1
    tsleep = 0.6
    if t2<tsleep:
        time.sleep(tsleep-t2)
    return df

def get_games_box(game_ids, fun, name):
    df_ap1, df_ap2 = [], []
    for game_id in tqdm(game_ids):
        for attempt in range(2):
            try:
                df0 = get_game_box(game_id, fun)
                if name != "Matchup":
                    df1 = df0[1]
                    df_ap1.append(df1)
                df2 = df0[0]
                df_ap2.append(df2)
                break
            except Exception as error:
                if attempt == 0:
                    print(error)
    return df_ap1, df_ap2



def update_boxscores_idv_PS(season, fun, name):
    game_ids, dfr1, dfr2 = get_gameids_PS(season, name)
    try:
        df_ap1, df_ap2 = get_games_box(game_ids, fun, name)
        if name != "Matchup":
            df1 = pd.concat(df_ap1)
            df3 = pd.concat([dfr1, df1])
            df3["gameId"] = df3["gameId"].astype(int)
            df3 = df3.sort_values(by=["gameId"]).reset_index(drop=True)
            df3.to_parquet(
                box_DIR + "NBA_Box_T_" + name + "_" + season + "_PS.parquet"
            )
        df2 = pd.concat(df_ap2)
        df4 = pd.concat([dfr2, df2])
        df4["gameId"] = df4["gameId"].astype(int)
        df4 = df4.sort_values(by=["gameId"]).reset_index(drop=True)
        df4.to_parquet(box_DIR+ "NBA_Box_P_"+ name + "_"+ season+ "_PS.parquet")
    except Exception as error:
        print(error)
        pass

In [ ]:
boxscores = [
    # 1996
    {
        "name": "Trad",
        "fun": boxscoretraditionalv3.BoxScoreTraditionalV3,
    },
    {
        "name": "Adv",
        "fun": boxscoreadvancedv3.BoxScoreAdvancedV3,
    },
    {
        "name": "4Factor",
        "fun": boxscorefourfactorsv3.BoxScoreFourFactorsV3,
    },
    {
        "name": "Misc",
        "fun": boxscoremiscv3.BoxScoreMiscV3,
    },
    {
        "name": "Scoring", 
        "fun": boxscorescoringv3.BoxScoreScoringV3
    },
    {
        "name":"Usage",
        "fun": boxscoreusagev3.BoxScoreUsageV3
    },
    # 2013
    {
        "name": "Track",
        "fun": boxscoreplayertrackv3.BoxScorePlayerTrackV3,
    },
    {
        "name": "Hustle",
        "fun": boxscorehustlev2.BoxScoreHustleV2,
    },
    # 2017
    {
        "name": "Def",
        "fun": boxscoredefensivev2.BoxScoreDefensiveV2
    },
    {
        "name": "Matchup",
        "fun": boxscorematchupsv3.BoxScoreMatchupsV3
    },
]

In [ ]:
for year in tqdm(range(2024,2025,1)):
    stats = leaguegamelog.LeagueGameLog(
        player_or_team_abbreviation="P",
        season=year,
        season_type_all_star="Playoffs",
    )
    df = stats.get_data_frames()[0]
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y-%m-%d")
    df.to_parquet(box_DIR + "NBA_Box_P_" + "Base" + "_" + str(year) + "_PS.parquet")
    stats = leaguegamelog.LeagueGameLog(
        player_or_team_abbreviation="T",
        season=year,
        season_type_all_star="Playoffs",
    )
    df = stats.get_data_frames()[0]
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], format="%Y-%m-%d")
    df.to_parquet(box_DIR + "NBA_Box_T_" + "Base" + "_" + str(year) + "_PS.parquet")
    time.sleep(1)
    for boxscore in boxscores:
        print(boxscore["name"])
        session = requests.Session()
        session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:72.0) Gecko/20100101 Firefox/72.0'
        })
        NBAStatsHTTP.set_session(session)
        update_boxscores_idv_PS(str(year),boxscore["fun"],boxscore["name"])